# Défi quotidien : Prompting précis pour le contrôle et la qualité de la sortie

Ce notebook reprend le défi en 5 étapes (contrôle du ton/format/longueur, évaluation, détection d'hallucinations, paraphrase, extraction de citation).



## Contexte

Texte original de la politique interne, trop formel et rempli de jargon, qui doit être transformé en contenu de micro-apprentissage clair pour une newsletter interne.

In [1]:
# On stocke le texte original de la politique dans une variable,
# afin de pouvoir le réutiliser facilement dans chaque prompt ci-dessous
original_text = """Les employés doivent s’assurer que tout accès à distance aux systèmes internes est établi via le VPN sécurisé approuvé. En aucun cas les connexions non sécurisées ou les appareils personnels sans protection de terminaux ne doivent être utilisés pour accéder à des données propriétaires ou des communications sensibles."""

print(original_text)

Les employés doivent s’assurer que tout accès à distance aux systèmes internes est établi via le VPN sécurisé approuvé. En aucun cas les connexions non sécurisées ou les appareils personnels sans protection de terminaux ne doivent être utilisés pour accéder à des données propriétaires ou des communications sensibles.


## Étape 1 : Créer une invite qui contrôle le ton, le format et la longueur de sortie

**Tâches**
- Réécrire le texte sur un ton amical et clair.
- Utiliser des puces pour organiser l'information.
- Paraphraser l'original, sans citation directe.
- Rester sous les 75 mots au total.

In [2]:
# On construit le prompt de l'étape 1 en combinant 4 contraintes :
# ton amical, format en puces, paraphrase (pas de citation), limite de 75 mots
prompt_step1 = f"""En utilisant le texte suivant, réécris-le sur un ton amical et clair. Utilise des puces pour organiser l'information. Paraphrase l'original, évitant les citations directes. Reste sous les 75 mots au total.

Texte d'entrée :
{original_text}
"""

print(prompt_step1)

En utilisant le texte suivant, réécris-le sur un ton amical et clair. Utilise des puces pour organiser l'information. Paraphrase l'original, évitant les citations directes. Reste sous les 75 mots au total.

Texte d'entrée :
Les employés doivent s’assurer que tout accès à distance aux systèmes internes est établi via le VPN sécurisé approuvé. En aucun cas les connexions non sécurisées ou les appareils personnels sans protection de terminaux ne doivent être utilisés pour accéder à des données propriétaires ou des communications sensibles.



## Étape 2 : Évaluer la sortie

**Tâche** : une fois la réponse générée par le modèle, on l'examine selon 6 critères : pertinence, clarté, structure, ton, longueur, exactitude factuelle.

Comme nous n'avons pas de connexion directe à un modèle dans ce notebook, on simule ci-dessous une réponse plausible que produirait un bon LLM avec `prompt_step1`, afin de pouvoir réellement appliquer la grille d'évaluation (plutôt que de simplement décrire les critères sans les utiliser).

In [3]:
# Réponse simulée que produirait un LLM avec le prompt de l'étape 1
# (ton amical, puces, paraphrase, sous les 75 mots)
reponse_simulee_etape1 = """• Utilisez toujours le VPN sécurisé approuvé pour vous connecter à distance aux systèmes internes.
• Évitez les connexions non sécurisées ou les appareils personnels non protégés.
• Ces règles protègent nos données et nos communications sensibles."""

# On calcule le nombre de mots pour vérifier la contrainte de longueur (75 mots max)
nombre_de_mots_etape1 = len(reponse_simulee_etape1.split())

print(reponse_simulee_etape1)
print(f"\nNombre de mots : {nombre_de_mots_etape1}")

• Utilisez toujours le VPN sécurisé approuvé pour vous connecter à distance aux systèmes internes.
• Évitez les connexions non sécurisées ou les appareils personnels non protégés.
• Ces règles protègent nos données et nos communications sensibles.

Nombre de mots : 37


In [4]:
# Évaluation de la réponse simulée selon les 6 critères demandés dans la consigne
evaluation_etape1 = {
    "pertinence": "Oui, la sortie reprend fidèlement le message central : utiliser le VPN approuvé et éviter les connexions/appareils non sécurisés.",
    "clarte": "Oui, le message est simple et facile à comprendre pour un public non technique, sans jargon.",
    "structure": "Oui, l'information est présentée en 3 puces claires et courtes.",
    "ton": "Oui, le ton est direct mais amical, adapté à une communication interne.",
    # on réutilise le nombre de mots calculé juste au-dessus pour vérifier automatiquement la contrainte de longueur
    "longueur_respectee": nombre_de_mots_etape1 <= 75,
    "exactitude_factuelle": "Oui, aucun détail de sécurité n'a été inventé ni supprimé : le VPN approuvé et l'interdiction des connexions/appareils non sécurisés sont bien conservés."
}

for critere, resultat in evaluation_etape1.items():
    print(f"{critere} : {resultat}")

pertinence : Oui, la sortie reprend fidèlement le message central : utiliser le VPN approuvé et éviter les connexions/appareils non sécurisés.
clarte : Oui, le message est simple et facile à comprendre pour un public non technique, sans jargon.
structure : Oui, l'information est présentée en 3 puces claires et courtes.
ton : Oui, le ton est direct mais amical, adapté à une communication interne.
longueur_respectee : True
exactitude_factuelle : Oui, aucun détail de sécurité n'a été inventé ni supprimé : le VPN approuvé et l'interdiction des connexions/appareils non sécurisés sont bien conservés.


## Étape 3 : Détecter et atténuer les hallucinations

**Tâche** : si le modèle ajoute du contenu qui n'existe pas dans le texte d'entrée (nouvelles règles, nouvelles technologies...), on doit le détecter, puis réécrire le prompt pour empêcher cela.

Pour illustrer concrètement le problème, on simule ci-dessous une réponse "à risque" qu'un modèle mal contraint pourrait produire, avec du contenu inventé.

In [5]:
# Exemple de réponse simulée où le modèle "hallucine" : il invente une technologie
# et une exigence qui n'apparaissent nulle part dans le texte original
reponse_hallucinee_exemple = """• Utilisez le VPN sécurisé approuvé pour l'accès à distance.
• Installez notre nouvel antivirus XDR-Secure sur tous les appareils.
• Activez l'authentification biométrique avant chaque connexion."""

print(reponse_hallucinee_exemple)

• Utilisez le VPN sécurisé approuvé pour l'accès à distance.
• Installez notre nouvel antivirus XDR-Secure sur tous les appareils.
• Activez l'authentification biométrique avant chaque connexion.


In [6]:
# On identifie précisément le contenu halluciné : des éléments qui ne figurent
# nulle part dans "original_text" et qui ont donc été inventés par le modèle
contenu_hallucine_identifie = [
    "'antivirus XDR-Secure' : ce nom de produit n'existe pas dans le texte d'entrée.",
    "'authentification biométrique' : cette exigence n'est pas mentionnée dans la politique originale."
]

for element in contenu_hallucine_identifie:
    print(f"- {element}")

- 'antivirus XDR-Secure' : ce nom de produit n'existe pas dans le texte d'entrée.
- 'authentification biométrique' : cette exigence n'est pas mentionnée dans la politique originale.


In [7]:
# Version révisée du prompt de l'étape 1, qui ajoute une contrainte stricte
# pour empêcher le modèle d'inventer de nouvelles règles ou technologies
prompt_step3 = f"""En utilisant le texte suivant, réécris-le sur un ton amical et clair. Utilise des puces pour organiser l'information. Paraphrase l'original, évitant les citations directes. Reste sous les 75 mots au total.

Ne fais que paraphraser le contenu du texte d'entrée. N'ajoute aucune nouvelle recommandation, règle ou technologie qui n'y figure pas déjà.

Texte d'entrée :
{original_text}
"""

print(prompt_step3)

En utilisant le texte suivant, réécris-le sur un ton amical et clair. Utilise des puces pour organiser l'information. Paraphrase l'original, évitant les citations directes. Reste sous les 75 mots au total.

Ne fais que paraphraser le contenu du texte d'entrée. N'ajoute aucune nouvelle recommandation, règle ou technologie qui n'y figure pas déjà.

Texte d'entrée :
Les employés doivent s’assurer que tout accès à distance aux systèmes internes est établi via le VPN sécurisé approuvé. En aucun cas les connexions non sécurisées ou les appareils personnels sans protection de terminaux ne doivent être utilisés pour accéder à des données propriétaires ou des communications sensibles.



## Étape 4 : Paraphraser — plongée approfondie

**Tâches**
- Paraphraser le paragraphe pour un public junior de stagiaires.
- Utiliser un langage clair et des phrases courtes.
- Ne pas dépasser 4 points clés.
- Éviter tout jargon d'entreprise ou juridique.
- Garder un ton solidaire et informatif.

In [8]:
# Prompt de l'étape 4 : même texte source, mais adapté à un public de stagiaires juniors
prompt_step4 = f"""En utilisant le texte suivant, paraphrase le paragraphe de politique pour un public junior de stagiaires, en utilisant un langage clair et des phrases courtes.

N'utilise pas plus de 4 points clés. Évite tout jargon d'entreprise ou juridique. Garde un ton solidaire et informatif.

Texte d'entrée :
{original_text}
"""

print(prompt_step4)

En utilisant le texte suivant, paraphrase le paragraphe de politique pour un public junior de stagiaires, en utilisant un langage clair et des phrases courtes.

N'utilise pas plus de 4 points clés. Évite tout jargon d'entreprise ou juridique. Garde un ton solidaire et informatif.

Texte d'entrée :
Les employés doivent s’assurer que tout accès à distance aux systèmes internes est établi via le VPN sécurisé approuvé. En aucun cas les connexions non sécurisées ou les appareils personnels sans protection de terminaux ne doivent être utilisés pour accéder à des données propriétaires ou des communications sensibles.



## Étape 5 : Variante d'extraction de citation

**Tâche** : parfois, citer directement est plus efficace que paraphraser. On rédige un prompt qui extrait une citation directe du texte original capturant la politique de sécurité centrale, puis on répond aux 2 questions posées.

In [9]:
# Prompt qui demande au modèle d'extraire une citation directe (et non une paraphrase)
prompt_step5_quote = f"""Extrais une citation directe du texte original suivant qui capture la politique de sécurité centrale.

Texte d'entrée :
{original_text}
"""

print(prompt_step5_quote)

Extrais une citation directe du texte original suivant qui capture la politique de sécurité centrale.

Texte d'entrée :
Les employés doivent s’assurer que tout accès à distance aux systèmes internes est établi via le VPN sécurisé approuvé. En aucun cas les connexions non sécurisées ou les appareils personnels sans protection de terminaux ne doivent être utilisés pour accéder à des données propriétaires ou des communications sensibles.



**Réponses aux questions :**

1. **Dans quel type de communication interne citer serait-il plus approprié que de paraphraser ?**

   Citer directement est préférable quand la précision juridique ou la fidélité exacte à la formulation originale est essentielle : documents officiels de politique, communications à la direction sur des points de conformité stricts, formations juridiques, ou rappels de clauses spécifiques où chaque mot compte pour éviter toute mauvaise interprétation.

2. **Quand le devis (la citation) pourrait-il représenter un risque ?**

   Citer directement peut poser problème si le langage original est trop technique ou complexe pour le public visé, ce qui peut créer de la confusion. De plus, une citation sortie de son contexte, sans explication, peut sembler trop rigide et pousser les employés à chercher des échappatoires à la lettre du texte plutôt qu'à en respecter l'esprit.